# Transformer Language Model Architecture

A language model takes as input a batched sequence of integer token IDs (i.e., `torch.Tensor` of shape
(`batch_size`, `sequence_length`)), and returns a (batched) normalized probability distribution over the
vocabulary (i.e., a PyTorch Tensor of shape (`batch_size`, `sequence_length`, `vocab_size`)), where the
predicted distribution is over the next token for each input token. 

When training the language model, we use these next-token predictions to calculate the cross-entropy loss between the actual next token and the predicted next token. When generating text from the language model during inference, we take the predicted next-token distribution from the final time step (i.e., the last item in the sequence) to generate the next token in the sequence (e.g., by taking the token with the highest probability, sampling from the distribution, etc.), add the generated token to the input sequence, and repeat.

In this project, we will build this Transformer language model from scratch.

## Linear and Embedding Modules

### Parameter Initialization

Pre-norm transformers are unusually robust to initializations, but they can still have a significant impact on training speed and convergence.

For now, use these approximate initializations (Normal Distribution here refer to the classic Gaussian bell curve distribution):

(a) Linear weights (e.g., in Feed Forward Neural Nets): $N(\mu = 0, \sigma^2 = \frac{2}{d_{in}+d_{out}})$, truncated at $[-3 \sigma, 3 \sigma]$, where $d_{in}$ and $d_{out}$ refer to the input and output dimensions respectively.

(b) Embedding (in the LLM context): $N(\mu = 0, \sigma^2 = 1)$, truncated at $[-3, 3]$.

(c) RMSNorm (in the LLM context): $\mathbb{1}$ -- uniformly 1's.

You should use `torch.nn.init.trunc_normal_` to initialize the truncated normal weights.

### Linear Module

Following most modern LLMs, we will not include a bias term.

#### Coding task for Linear Module:

Implement a `Linear` Python class that inherits from `torch.nn.Module` and performs a linear transformation. Your implementation should follow the following interface. This is intended to resemble the interface of PyTorch’s built-in `nn.Linear` module, except for not having a bias argument or parameter.

- `def __init__(self, in_features, out_features, device=None, dtype=None)` 
  - Construct a linear transformation module. This function should accept the following parameters:
    - `in_features`: `int` 
      final dimension of the input
    - `out_features`: `int` 
      final dimension of the output
    - `device`: `torch.device | None = None` 
      Device to store the parameters on
    - `dtype`: `torch.dtype | None = None` 
      Data type of the parameters

- `def forward(self, x: torch.Tensor) -> torch.Tensor` 
  - Apply the linear transformation to the input.

Make sure to:

(i) subclass `nn.Module`

(ii) call the superclass constructor

(iii) construct and store your parameter as W, putting it in an `nn.Parameter`. NOTE: $W \in \mathbb{R}^{d_{out} \times d_{in}}$ is in **row-major** form, where each row corresponds to one output feature. In `forward`, compute the transformation as $x W^T$ (equivalently, `x @ W.T`) over the final input dimension, preserving any leading dimensions of `x`.

(iv) do **not** use `nn.Linear` or `nn.functional.linear`

For initializations, use the settings from above along with `torch.nn.init.trunc_normal_` to initialize the weights.

To test your `Linear` module, implement the test adapter at [adapters.run_linear]. The adapter should load the given weights into your `Linear` module. You can use `Module.load_state_dict` for this purpose. Then, run `uv run pytest -k test_linear` and check that all unit tests pass.